# LLM 생성 테스트는 코드 변화를 얼마나 버티는가

이 노트북은 진행 중인 실험의 일부를 줄여서 직접 실행해 보는 강의 자료입니다.

- 대상: json-schema-validator (Java, JSON Schema 검증 라이브러리)
- 비교 집단: 개발자가 작성한 테스트(Dev), Ouédraogo 방식으로 생성한 테스트, KTester 방식으로 생성한 테스트
- 진행: 시점 t에서 테스트를 한 번 생성하고, 이후 commit 몇 개에서 수정 없이 실행해 Dev 테스트와 비교합니다.

이 노트북은 **Google Colab** 기준으로 만들었습니다. 위에서부터 순서대로 실행하세요.

## 1. 환경 준비

테스트를 생성하고 실행하려면 아래 도구가 필요합니다. 실제 실험도 같은 구성으로 테스트를 실행합니다.

| 도구 | 하는 일 |
|---|---|
| JDK 17 | Java 코드 컴파일과 실행 |
| Maven | 대상 프로젝트 빌드, commit마다 의존성(classpath) 받기 |
| JUnit 콘솔 런처 | 생성된 테스트를 메서드 단위로 실행하고 결과 수집 |
| Mockito | 생성된 테스트가 쓰는 mock 라이브러리 (KTester 프롬프트가 JUnit 5와 Mockito 사용을 지시함) |
| JaCoCo | 테스트가 production 코드의 어느 줄을 실행했는지 측정 |
| google-genai | Gemini API 호출 |

### 1-1. JDK와 Maven 설치

Colab에는 Java가 없거나 버전이 다를 수 있어서 JDK 17과 Maven을 설치합니다. 1~2분 걸립니다.

In [ ]:
import os, sys, subprocess, shutil, pathlib

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    print("주의: 이 노트북은 Colab 기준입니다. 다른 환경에서는 JDK 17과 Maven이 미리 설치돼 있어야 합니다.")

def sh(cmd, check=True, quiet=False):
    """Run a shell command and return its output."""
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if not quiet and out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return out

JDK17 = "/usr/lib/jvm/java-17-openjdk-amd64"
if IN_COLAB and not (os.path.isdir(JDK17) and shutil.which("mvn")):
    sh("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless maven > /dev/null", quiet=True)

if os.path.isdir(JDK17):
    os.environ["JAVA_HOME"] = JDK17
    os.environ["PATH"] = f"{JDK17}/bin:" + os.environ["PATH"]

print(sh("java -version 2>&1 | grep -i 'version'", quiet=True))
print(sh("mvn -v 2>&1 | head -1", quiet=True))

### 1-2. 작업 폴더와 테스트 실행용 jar 받기

작업 폴더를 만들고, 테스트 실행에 쓸 jar를 Maven으로 받습니다. 필요한 jar 목록을 작은 `pom.xml` 하나에 적고 Maven이 의존 jar까지 함께 받게 합니다(Mockito는 byte-buddy, objenesis가 필요합니다).

In [ ]:
WORK = pathlib.Path("/content/lecture" if IN_COLAB else os.path.expanduser("~/lecture-work"))
LIB = WORK / "lib"
for d in [WORK, LIB, WORK / "tools"]:
    d.mkdir(parents=True, exist_ok=True)

TOOLS_POM = """<project xmlns="http://maven.apache.org/POM/4.0.0">
  <modelVersion>4.0.0</modelVersion>
  <groupId>lecture</groupId><artifactId>tools</artifactId><version>1</version>
  <dependencies>
    <dependency><groupId>org.junit.platform</groupId><artifactId>junit-platform-console-standalone</artifactId><version>1.9.3</version></dependency>
    <dependency><groupId>org.mockito</groupId><artifactId>mockito-core</artifactId><version>5.11.0</version></dependency>
    <dependency><groupId>org.mockito</groupId><artifactId>mockito-junit-jupiter</artifactId><version>5.11.0</version>
      <exclusions><exclusion><groupId>org.junit.jupiter</groupId><artifactId>*</artifactId></exclusion></exclusions></dependency>
    <dependency><groupId>org.jacoco</groupId><artifactId>org.jacoco.agent</artifactId><version>0.8.12</version><classifier>runtime</classifier></dependency>
    <dependency><groupId>org.jacoco</groupId><artifactId>org.jacoco.cli</artifactId><version>0.8.12</version><classifier>nodeps</classifier>
      <exclusions><exclusion><groupId>*</groupId><artifactId>*</artifactId></exclusion></exclusions></dependency>
  </dependencies>
</project>"""
(WORK / "tools" / "pom.xml").write_text(TOOLS_POM)
sh(f"cd {WORK/'tools'} && mvn -q dependency:copy-dependencies -DoutputDirectory={LIB}")

def jar(prefix):
    return str(next(LIB.glob(prefix + "*.jar")))

LAUNCHER = jar("junit-platform-console-standalone")
JACOCO_AGENT = jar("org.jacoco.agent")
JACOCO_CLI = jar("org.jacoco.cli")
TEST_LIBS = [str(p) for p in sorted(LIB.glob("*.jar")) if "jacoco" not in p.name]

for p in sorted(LIB.glob("*.jar")):
    print(p.name)

### 1-3. 테스트 실행 도구 점검

이후 단계에서 쓸 실행 순서를 작은 예제로 먼저 확인합니다. 실제 실험에서 생성된 테스트를 실행하는 순서와 같습니다.

1. production 클래스와 테스트 클래스를 `javac`로 컴파일합니다.
2. JUnit 콘솔 런처로 테스트를 실행합니다. 이때 JaCoCo 에이전트를 붙여 실행된 줄을 기록합니다.
3. JaCoCo 기록에서 줄 단위 coverage를 읽습니다.

예제의 `PriceCalculator`는 `DiscountService`에 의존하고, 테스트는 Mockito로 `DiscountService`를 대신합니다.

In [ ]:
import csv, io, json, re, xml.etree.ElementTree as ET

def compile_java(src_dir, out_dir, classpath):
    """javac all .java files under src_dir. Returns (ok, error_text)."""
    files = [str(p) for p in pathlib.Path(src_dir).rglob("*.java")]
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["javac", "-nowarn", "-encoding", "UTF-8", "-d", str(out_dir),
                        "-cp", os.pathsep.join(classpath)] + files,
                       text=True, capture_output=True)
    return r.returncode == 0, r.stderr

def run_tests(classpath, select_class, report_dir, jacoco_exec=None, timeout=300):
    """Run one test class with the JUnit console launcher.
    Returns {test method: 'pass'|'fail'} parsed from the XML report."""
    report_dir = pathlib.Path(report_dir)
    shutil.rmtree(report_dir, ignore_errors=True)
    cmd = ["java"]
    if jacoco_exec:
        cmd.append(f"-javaagent:{JACOCO_AGENT}=destfile={jacoco_exec},append=true")
    cmd += ["-jar", LAUNCHER, "--disable-banner", "--details=none",
            "--class-path", os.pathsep.join(classpath),
            "--select-class", select_class, "--reports-dir", str(report_dir)]
    subprocess.run(cmd, text=True, capture_output=True, timeout=timeout)
    results = {}
    for xml in report_dir.glob("*.xml"):
        for tc in ET.parse(xml).getroot().iter("testcase"):
            name = tc.get("name").split("(")[0]
            bad = tc.find("failure") is not None or tc.find("error") is not None
            results[name] = "fail" if bad else "pass"
    return results

def line_coverage(jacoco_exec, classes_dir, source_dir):
    """Return {(class, line): covered?} for every executable line, from a JaCoCo exec file."""
    xml_path = pathlib.Path(jacoco_exec).with_suffix(".xml")
    sh(f"java -jar {JACOCO_CLI} report {jacoco_exec} --classfiles {classes_dir} "
       f"--sourcefiles {source_dir} --xml {xml_path}", quiet=True)
    lines = {}
    root = ET.parse(xml_path).getroot()
    for pkg in root.iter("package"):
        for sf in pkg.iter("sourcefile"):
            for ln in sf.iter("line"):
                lines[(pkg.get("name") + "/" + sf.get("name"), int(ln.get("nr")))] = int(ln.get("ci")) > 0
    return lines

# --- tiny example ---------------------------------------------------------
demo = WORK / "demo"
shutil.rmtree(demo, ignore_errors=True)
(demo / "src/shop").mkdir(parents=True)
(demo / "test/shop").mkdir(parents=True)
(demo / "src/shop/DiscountService.java").write_text("""package shop;
public interface DiscountService { int percentFor(String customer); }
""")
(demo / "src/shop/PriceCalculator.java").write_text("""package shop;
public class PriceCalculator {
    private final DiscountService discounts;
    public PriceCalculator(DiscountService discounts) { this.discounts = discounts; }
    public int finalPrice(String customer, int price) {
        if (price < 0) {
            throw new IllegalArgumentException("negative price");
        }
        int pct = discounts.percentFor(customer);
        return price - price * pct / 100;
    }
}
""")
(demo / "test/shop/PriceCalculatorTest.java").write_text("""package shop;
import org.junit.jupiter.api.Test;
import static org.junit.jupiter.api.Assertions.*;
import static org.mockito.Mockito.*;
class PriceCalculatorTest {
    @Test void appliesDiscount() {
        DiscountService d = mock(DiscountService.class);
        when(d.percentFor("kim")).thenReturn(10);
        assertEquals(900, new PriceCalculator(d).finalPrice("kim", 1000));
    }
    @Test void wrongExpectation() {
        DiscountService d = mock(DiscountService.class);
        when(d.percentFor("lee")).thenReturn(0);
        assertEquals(999, new PriceCalculator(d).finalPrice("lee", 1000));
    }
}
""")

ok, err = compile_java(demo / "src", demo / "classes", TEST_LIBS)
ok_t, err_t = compile_java(demo / "test", demo / "test-classes", TEST_LIBS + [str(demo / "classes")])
print("컴파일:", "production", ok, "/ test", ok_t)

exec_file = demo / "jacoco.exec"
exec_file.unlink(missing_ok=True)
res = run_tests(TEST_LIBS + [str(demo / "classes"), str(demo / "test-classes")],
                "shop.PriceCalculatorTest", demo / "report", jacoco_exec=exec_file)
print("실행 결과:", res)

cov = line_coverage(exec_file, demo / "classes", demo / "src")
calc = {ln: c for (f, ln), c in cov.items() if f.endswith("PriceCalculator.java")}
print("PriceCalculator 실행 줄:", sorted(ln for ln, c in calc.items() if c))
print("PriceCalculator 실행 안 된 줄:", sorted(ln for ln, c in calc.items() if not c))

위 결과에서 볼 것:

- `appliesDiscount`는 통과하고 `wrongExpectation`은 실패합니다. 기대값 999가 틀렸기 때문입니다.
- 음수 가격을 던지는 7번째 줄은 어느 테스트도 실행하지 않아 "실행 안 된 줄"에 나옵니다.

이후 단계에서는 이 세 함수(`compile_java`, `run_tests`, `line_coverage`)로 생성된 테스트와 Dev 테스트를 똑같이 실행합니다.

### 1-4. Gemini API 키 설정

수강생마다 자기 API 키를 씁니다. 키는 [Google AI Studio](https://aistudio.google.com/apikey)에서 발급합니다.

Colab 왼쪽의 열쇠 모양(Secrets)에 이름 `GEMINI_API_KEY`로 키를 저장하고 이 노트북의 접근을 허용하세요. 저장하지 않았으면 아래 셀이 입력 칸을 띄웁니다. 키는 노트북 파일에 남지 않습니다.

In [ ]:
sh("pip install -q google-genai", quiet=True)
from google import genai
import google.genai.errors
from getpass import getpass

API_KEY = None
if IN_COLAB:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = None
API_KEY = API_KEY or os.environ.get("GEMINI_API_KEY") or getpass("Gemini API key: ")

MODEL = "gemini-3.8-flash"   # 사용할 모델. 바꾸려면 이 줄만 고치면 됩니다.
client = genai.Client(api_key=API_KEY)

try:
    reply = client.models.generate_content(model=MODEL, contents="Reply with the single word OK.")
    print("모델:", MODEL, "/ 응답:", reply.text.strip())
except genai.errors.ClientError as e:
    print("호출 실패:", e)
    print("\n이 키로 쓸 수 있는 flash 모델 목록입니다. 위 MODEL 줄을 이 중 하나로 바꿔 다시 실행하세요.")
    for m in client.models.list():
        if "flash" in m.name and "generateContent" in (m.supported_actions or []):
            print(" ", m.name.removeprefix("models/"))